# **FLUXO DE MODELAGEM DE REGRESSÃO LINEAR**
### *Passos sugeridos*
---



### **Bibliotecas**

In [ ]:
import pandas as pd                                                             # Manipulação de dados
import numpy as np                                                              # Realização de cálculos específicos
import matplotlib.pyplot as plt                                                 # Visualização de dados
import seaborn as sns                                                           # Visualização de dados
import math                                                                     # Funções matemáticas
from sklearn.compose import ColumnTransformer                                   # Transformação de colunas
import statsmodels.api as sm                                                    # Ajuste de modelos de regressão
from statsmodels.stats.outliers_influence import variance_inflation_factor      # Cálculo do VIF
from sklearn.preprocessing import OneHotEncoder                                 # Codificação one-hot

### **Leitura da base de dados**

In [ ]:
dados = pd.read_table("nome_da_base_de_dados.txt",  # Substitua pelo nome da base de dados
                      sep="\t",                     # Ajuste o separador de colunas, se necessário
                      decimal=".",                  # Ajuste o separador de decimal, se necessário
                      header=0)

### **Visualização da base de dados**

In [ ]:
dados.head()

### **Dimensões da base de dados**

In [ ]:
dados.shape

### **Tipos das colunas da base de dados**

In [ ]:
dados.dtypes

### **Etapa 1: Tratamento de valores ausentes, especificação de variáveis e análise bivariada**

In [ ]:
# Verificando a quantidade de valores ausentes
dados.isna().sum()

In [ ]:
# Sugestão de tratamento de valores ausentes em uma variável explicativa quantitativa, por meio da criação de faixas
# dados['NOME_VARIAVEL'] = pd.to_numeric(dados['NOME_VARIAVEL'])
# dados['NOME_VARIAVEL_CAT'] = pd.qcut(dados['NOME_VARIAVEL'], q=10, duplicates='drop')
# dados['NOME_VARIAVEL_CAT'] = dados['NOME_VARIAVEL_CAT'].cat.add_categories('NA').fillna('NA')
# dados['NOME_VARIAVEL_CAT'].value_counts(dropna=False)

*Lista de nomes das variáveis explicativas, separando em quantitativas e qualitativas*

In [ ]:
# Variáveis explicativas quantitativas (deixar vazio [] caso não haja nenhuma)
lista_X_quanti = ['NOME_VARIAVEL_1',
                  'NOME_VARIAVEL_2',
                  'NOME_VARIAVEL_3',
                  '...']

# Variáveis explicativas qualitativas (deixar vazio [] caso não haja nenhuma)
lista_X_quali = ['NOME_VARIAVEL_1',
                 'NOME_VARIAVEL_2',
                 'NOME_VARIAVEL_3',
                 '...']

*Objetos para variável resposta (y) e explicativas (X)*

In [ ]:
y = dados['NOME_VARIAVEL_RESPOSTA']
X = dados[lista_X_quanti + lista_X_quali]

*Análise bivariada: gráficos de boxplot para variáveis explicativas qualitativas versus variável resposta*

In [ ]:
if lista_X_quali:
    n = len(lista_X_quali)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows=nrows,
                             ncols=ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for i, var in enumerate(lista_X_quali):
        sns.boxplot(y=dados[var],
                    x=y,
                    ax=axes[i],
                    orient='h',
                    color='darkturquoise')
        axes[i].set_title(f'{y.name} vs. {var}')
        axes[i].tick_params(axis='x', rotation=0)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout(h_pad=2, w_pad=2)
    plt.show()

*Análise bivariada: gráficos de dispersão para variáveis explicativas quantitativas versus variável resposta*

In [ ]:
if lista_X_quanti:
    n = len(lista_X_quanti)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows=nrows,
                             ncols=ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for i, var in enumerate(lista_X_quanti):
        sns.scatterplot(x=dados[var],
                        y=y,
                        ax=axes[i],
                        color='darkturquoise',
                        alpha=0.5)
        axes[i].set_title(f'{y.name} vs. {var}')
        axes[i].set_ylabel("")

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

*Análise bivariada: matriz de correlações lineares de Pearson para variáveis explicativas quantitativas*

In [ ]:
matriz_cor = (pd.concat([dados[lista_X_quanti], y], axis=1)).corr().round(3)
matriz_cor

*Análise bivariada: gráfico de calor para representar a matriz de correlações*

In [ ]:
mask = np.tril(np.ones(matriz_cor.shape, dtype=bool))

plt.figure(figsize=(12, 6))
sns.heatmap(matriz_cor, mask=mask, annot=True, fmt=".3f", cmap="coolwarm")
plt.title("Matriz de Correlações Lineares de Pearson", fontsize=14)
plt.show()

### **Etapa 2: Pré-processamento de variáveis explicativas**

*Definição da função de pré-processamento: Codificação one-hot para variáveis qualitativas*

In [ ]:
preprocessador = ColumnTransformer(transformers=[
    ("quali", OneHotEncoder(sparse_output=False, drop="first", handle_unknown='ignore'), lista_X_quali)
],
    remainder='passthrough'
)

*Criação e aplicação do pré-processamento*

In [ ]:
X_tratada = preprocessador.fit_transform(X)
if lista_X_quali:
    nomes_quali = list(preprocessador.named_transformers_['quali'].get_feature_names_out(lista_X_quali))
else:
    nomes_quali = []
nomes_variaveis = nomes_quali + list(lista_X_quanti)

In [ ]:
X_tratada = pd.DataFrame(X_tratada, columns=nomes_variaveis)
X_tratada.head()

### **Etapa 3: Construção do modelo de regressão linear**

*Ajuste do modelo 1*

In [ ]:
modelo_1 = sm.OLS(y, sm.add_constant(X_tratada)).fit()
print(modelo_1.summary(alpha=0.10))

*Ajuste do modelo 2*

In [ ]:
# Excluindo a variável 'XYZ'
X_tratada = X_tratada.drop(['XYZ'], axis=1)

In [ ]:
modelo_2 = sm.OLS(y, sm.add_constant(X_tratada)).fit()
print(modelo_2.summary())

*Ajuste do modelo 3*

In [ ]:
# Excluindo a variável 'ABC'
X_tratada = X_tratada.drop(['ABC'], axis=1)

In [ ]:
modelo_3 = sm.OLS(y, sm.add_constant(X_tratada)).fit()
print(modelo_3.summary())

*Demais modelos*

In [ ]:
# Ajuste demais modelos, se necessário

*Avaliação de colinearidade*

In [ ]:
modelo = modelo_X    # Digite o nome do objeto do modelo que deseja avaliar
vif = pd.DataFrame({
    "Variável": modelo.model.exog_names,
    "VIF": [variance_inflation_factor(modelo.model.exog, i) for i in range(modelo.model.exog.shape[1])]
})

print(vif)

### **Etapa 4: Qualidade de ajuste e propriedades dos resíduos**

*Coeficiente de determinação ajustado (R²)*

In [ ]:
modelo_X.rsquared_adj   # Altere o nome do objeto do modelo

*Valores preditos pelo modelo (y^)*

In [ ]:
y_hat = modelo_X.predict(sm.add_constant(X_tratada))   # Altere o nome do objeto do modelo

*Erro absoluto médio (MAE)*

In [ ]:
np.mean(np.abs(y - y_hat))

*Erro absoluto percentual médio (MAPE)*

In [ ]:
np.mean(np.abs((y - y_hat) / y))

*Erro quadrático médio em raiz (RMSE)*

In [ ]:
np.sqrt(np.mean((y - y_hat)**2))

*Resíduos do modelo (e)*

In [ ]:
resid = modelo_X.resid   # Altere o nome do objeto do modelo

*Histograma dos resíduos*

In [ ]:
sns.histplot(resid,
             bins=10,    # Altere a quantidade de bins conforme necessário
             color="darkturquoise",
             edgecolor="white",
             kde=True)
plt.title("Histograma dos Resíduos")
plt.xlabel("Resíduos")
plt.ylabel("Frequência")
plt.show()

*Q-Q plot dos resíduos*

In [ ]:
sm.qqplot(resid,
          line='45',
          fit=True,
          markerfacecolor='darkturquoise',
          markeredgecolor='darkturquoise',
          alpha=0.7)
plt.title("QQ-plot dos Resíduos")
plt.xlabel("Percentis Teóricos")
plt.ylabel("Percentis Amostrais")
plt.show()

*Gráfico de resíduos vs. valores preditos*

In [ ]:
plt.scatter(y_hat,
            resid,
            color="darkturquoise",
            alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.title("Resíduos vs. Valores Preditos")
plt.xlabel("Valores Preditos")
plt.ylabel("Resíduos")
plt.show()

*Gráfico de valores observados vs. valores preditos da resposta*

In [ ]:
plt.scatter(y,
            y_hat,
            color="darkturquoise",
            alpha=0.7)
plt.plot([y_hat.min(), y_hat.max()], [y_hat.min(), y_hat.max()], color='red', linestyle='--')
plt.title("Valores Observados vs. Valores Preditos")
plt.xlabel("Valores Observados")
plt.ylabel("Valores Preditos")
plt.show()

### **Etapa 5: Análise de observações influentes**

*Cálculo das distâncias de Cook*

In [ ]:
influencias = sm.OLS(y, sm.add_constant(X_tratada)).fit().get_influence()
dist_cook = influencias.cooks_distance[0]

*Especificação de limite para identificação de pontos influentes*

In [ ]:
# Limite padrão (4/n)
limite = 4/dados.shape[0]

# OU

# Limite personalizado
# limite = ...

*Gráfico das distâncias de Cook*

In [ ]:
plt.stem(dist_cook,
         markerfmt="o",
         linefmt="gray",
         basefmt=" ")
plt.axhline(limite,
            color='red',
            linestyle='--')
plt.title("Distâncias de Cook")
plt.xlabel("Índice da Observação")
plt.ylabel("Distância de Cook")
plt.show()

*Reajuste do modelo e análise da qualidade de ajuste*

In [ ]:
# Separando pontos influentes, com base na distância de Cook acima do limite
influentes = np.where(dist_cook > limite)[0]

# Criação de novos objetos X e y excluindo as observações influentes
X_filtrada = X_tratada.loc[~dados.index.isin(influentes)]
y_filtrada = y.loc[~dados.index.isin(influentes)]

# Reajuste do modelo, excluindo as observações influentes
modelo_sem_influ = sm.OLS(y_filtrada, sm.add_constant(X_filtrada)).fit()
print(modelo_sem_influ.summary())

# Qualidade de ajuste
y_hat = modelo_sem_influ.predict(sm.add_constant(X_tratada))
print("R2 ajustado:", modelo_sem_influ.rsquared_adj)
print("MAE:", np.mean(np.abs(y - y_hat)))
print("MAPE:", np.mean(np.abs((y - y_hat) / y)))
print("RMSE:", np.sqrt(np.mean((y - y_hat)**2)))

### **Etapa 6: Utilização do modelo em nova base de dados**

*Especificação do modelo final*

In [ ]:
modelo_final = modelo_X   # Escolha o modelo final

*Leitura da nova base de dados*

In [ ]:
# Considerando a própria base de construção, mas pode ser substituída por uma nova, se houver
dados_novos = pd.read_table("nome_da_nova_base_de_dados.txt",  # Substitua pelo nome da base de dados
                            sep="\t",                          # Ajuste o separador de colunas, se necessário
                            dec=".",                           # Ajuste o separador de decimal, se necessário
                            header=0)

*Verificação de compatibilidade de nomes e tipos das variáveis, em relação à base de construção*

In [ ]:
# Comparação de nomes
dados.columns == dados_novos.columns

In [ ]:
# Comparação de tipos
dados.dtypes == dados_novos.dtypes

*Criação de objetos y e X*

In [ ]:
y_nova = dados['NOME_VARIAVEL_RESPOSTA']
X_nova = dados[lista_X_quanti + lista_X_quali]

*Aplicação do pré-processamento e criação de matriz X tratada*

In [ ]:
X_tratada_nova = preprocessador.transform(X_nova)  # Note que o mesmo objeto de pré-processamento deve ser utilizado, agora com 'transform' em vez de 'fit_transform'
X_tratada_nova = pd.DataFrame(X_tratada_nova, columns=nomes_variaveis)[X_tratada.columns]
X_tratada_nova = sm.add_constant(X_tratada_nova, has_constant='add')

In [ ]:
X_tratada_nova

*Aplicação do modelo final na nova base de dados*

In [ ]:
y_hat_nova = modelo_final.predict(X_tratada_nova)